[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Pi3X_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pi3X_Colab.ipynb)

# 🎯 Pi3X — Video-Native 3DGS with Permutation-Equivariance (ICLR 2026)

**[π³ / Pi3X](https://yyfz.github.io/pi3/)** is a feed-forward neural network for visual geometry reconstruction from Shanghai AI Lab + ZJU (ICLR 2026). Unlike every other 3DGS-from-images tool in this suite, **π³ is permutation-equivariant** — there is no fixed reference view. Input order is irrelevant, the model handles long videos without drift, and reconstruction quality stays stable on sequences of 10, 50, 200+ frames.

**Pi3X** is the engineering update of π³: smoother Convolutional Head, optional camera/pose/depth conditioning, continuous confidence prediction, and approximate metric scale.

Output is a metric point cloud + camera poses per frame, which we feed into `gsplat` to produce a real 3DGS scene.

* **Paper:** [arXiv 2507.13347](https://arxiv.org/abs/2507.13347) &nbsp;·&nbsp; **Project:** [yyfz.github.io/pi3/](https://yyfz.github.io/pi3/) &nbsp;·&nbsp; **Code:** [github.com/yyfz/Pi3](https://github.com/yyfz/Pi3) &nbsp;·&nbsp; **Weights (CC BY-NC-4.0):** [🤗&nbsp;yyfz233/Pi3X](https://huggingface.co/yyfz233/Pi3X)
* **Authors:** Yifan Wang, Jianjun Zhou, Haoyi Zhu, Wenzheng Chang, Yang Zhou, Zizun Li, Junyi Chen, Jiangmiao Pang, Chunhua Shen, Tong He (Shanghai AI Lab + Zhejiang University)
* **Conference:** ICLR 2026
* **License:** **Code = BSD 3-Clause** (commercial-OK). **Weights = CC BY-NC-4.0** (strictly non-commercial). The model card explicitly says: "Strictly Non-Commercial".

## How it differs from our other 3DGS notebooks

| Notebook | Inputs | Approach | License |
|---|---|---|---|
| **TripoSplat** | 1 image | Generative prior, feed-forward | MIT |
| **NoPoSplat** | 2-3 unposed photos | Pose-free, feed-forward | MIT (+ MASt3R backbone CC BY-NC-SA) |
| **Wild GS** | Video / image folder | MASt3R pose + 3DGS optimization | CC BY-NC-SA + INRIA non-commercial |
| **MapAnything** | Any 1+ images; optional poses/intrinsics/depth | Universal feed-forward transformer, then gsplat | **Apache 2.0** (commercial-OK) |
| **Pi3X (this)** | Video frames (any order, any count) | Permutation-equivariant feed-forward, then gsplat | **BSD-3 code + CC BY-NC-4.0 weights** (non-commercial) |

The standout property of Pi3X is **permutation-equivariance**: there is no reference view, no iterative pose refinement, and the model produces consistent reconstructions on **long videos** (50-200+ frames) where other feed-forward methods drift catastrophically. Give it a 2-minute phone video at 1 fps and you get a clean 3DGS scene in ~2-5 minutes total.

## Pipeline

```
video frames (or image folder)  — any number, any order
       ↓
Pi3X (π³) — permutation-equivariant feed-forward transformer, 1 forward pass
       ↓
metric point cloud + per-frame camera poses + intrinsics + depth
       ↓
write COLMAP-format (cameras.txt, images.txt, points3D.ply)
       ↓
gsplat (1-3 min training on T4)
       ↓
3DGS .ply (SOG/SPLAT/SPZ after SplatTransform_Colab STEP 3)
```

## Requirements
* **GPU:** NVIDIA, ≥ 6 GB VRAM (T4 15 GB works; A100/L4 recommended for 100+ frames)
* **RAM:** ≥ 12 GB
* **Disk:** ≈ 8 GB free (PyTorch + CUDA + ~5 GB Pi3X + working space)
* **Time on first run:** 5-8 min (PyTorch + π³ + safetensors download)
* **Time on subsequent runs:** 1-2 min (everything cached in your Drive)
* **Per-video runtime:** ~5-30 seconds for 100 frames at 224×224 on T4 (one forward pass)
* **Followed by gsplat:** +1-3 min depending on iteration count

## Where it fits in our 3DGS pipeline
```
Pi3X (this notebook, BSD-3 + CC BY-NC-4.0)
   →  .ply + COLMAP cameras
   →  SplatTransform_Colab STEP 3  →  SOG/SPLAT/SPZ/GLB
   →  Asset_Library_Browser_Colab
   →  Three.js / WebGPU game engine
```

> **⚠️ License note:** The Pi3X weights are **CC BY-NC-4.0** (strictly non-commercial). Use this notebook for research, personal projects, and non-commercial work. For commercial 3DGS-from-video, use **MapAnything** (Apache 2.0) instead.

In [ ]:
#@title STEP 1 — Install dependencies + clone repo
"""
• Detects GPU + CUDA, installs torch 2.5.1+cu121, torchvision 0.20.1, numpy 1.26.4
• git clone --depth=1 yyfz/Pi3 (BSD-3)
• pip install safetensors + huggingface_hub + the inference deps
• Pi3X model.safetensors + README cached to Drive
"""
import os, sys, time, json, subprocess, shutil, pathlib, traceback

print('='*72)
print('Pi3X — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda or "none"})')
    print(f'  CUDA avail   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — Pi3X is unusable on CPU')
except ImportError:
    torch = None
    print('  torch not yet installed (will be installed below)')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}  # info='Mount Google Drive and cache the 5 GB Pi3X model + intermediate datasets. Disable for ephemeral /content-only storage (model re-downloads each session).'
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/Pi3X')
    drive_root.mkdir(parents=True, exist_ok=True)
    print(f'  Drive cache  : {drive_root}')
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    os.environ['TORCH_HOME'] = str(drive_root / 'torch')
    os.environ['XDG_CACHE_HOME'] = str(drive_root / 'xdg')
else:
    drive_root = pathlib.Path('/content/_pi3x_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['TORCH_HOME'] = str(drive_root / 'torch')
    print(f'  Local cache  : {drive_root}  (lost on runtime disconnect)')

REPO_DIR = drive_root / 'Pi3'
t_total = time.time()

# 1. PyTorch ──────────────────────────────────────────────────
if torch is None or not torch.cuda.is_available() or not torch.__version__.startswith('2.5'):
    print('\n[1/5] Installing PyTorch 2.5.1 + cu121 ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
         'numpy==1.26.4',
         '--index-url', 'https://download.pytorch.org/whl/cu121'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('PyTorch install failed')
    print(f'  PyTorch installed in {time.time()-t0:.0f}s')
else:
    print(f'\n[1/5] PyTorch {torch.__version__} already installed')

# 2. Clone the Pi3 repo ───────────────────────────────────────
if not (REPO_DIR / '.git').exists():
    print(f'\n[2/5] Cloning yyfz/Pi3 into {REPO_DIR} ...')
    t0 = time.time()
    r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/yyfz/Pi3.git',
         str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  git clone failed:', r.stderr[-500:])
        raise RuntimeError('git clone failed')
    print(f'  cloned in {time.time()-t0:.0f}s')
else:
    print(f'\n[2/5] Reusing cached repo at {REPO_DIR}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 3. Install inference deps ───────────────────────────────────
print('\n[3/5] Installing inference deps ...')
t0 = time.time()
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     'safetensors', 'huggingface_hub', 'opencv-python-headless==4.10.0.84',
     'pillow-heif', 'einops', 'scipy', 'tqdm', 'matplotlib'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
print(f'  inference deps installed in {time.time()-t0:.0f}s')

# 4. Install gsplat for the 3DGS training step ───────────────
print('\n[4/5] Installing gsplat for 3DGS training ...')
t0 = time.time()
try:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         'gsplat==1.3.0', 'tyro', 'open3d'],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    print(f'  gsplat installed in {time.time()-t0:.0f}s')
except Exception as e:
    print(f'  gsplat install failed: {e}\n  (will retry in STEP 4)')

# 5. Download Pi3X model.safetensors from HuggingFace ─────────
print('\n[5/5] Downloading Pi3X from HuggingFace to Drive ...')
t0 = time.time()
from huggingface_hub import snapshot_download
model_dir = drive_root / 'Pi3X-weights'
if (model_dir / 'model.safetensors').exists() and \
   (model_dir / 'model.safetensors').stat().st_size > 4_000_000_000:
    print(f'  cached : yyfz233/Pi3X  ({(model_dir / "model.safetensors").stat().st_size//(1024*1024)} MB)')
else:
    print('  download : yyfz233/Pi3X/model.safetensors  (≈5 GB, 5-8 min)')
    snapshot_download(
        repo_id='yyfz233/Pi3X',
        local_dir=str(model_dir),
        allow_patterns=['model.safetensors', 'config.json', 'README.md'],
    )
    print(f'    -> {model_dir}')
print(f'  total setup time: {time.time()-t0:.0f}s')

print()
print('='*72)
print(f'  STEP 1 complete  (total {time.time()-t_total:.0f}s)')
print('  Next: run STEP 2 (imports + lazy model load)')
print('='*72)


In [ ]:
#@title STEP 2 — Imports & lazy Pi3X model load
"""
Imports the Pi3 package (one-time cost). Builds the model via
`Pi3X.from_pretrained(..., cache_dir=drive_root)`. The model is cached so
subsequent Gradio clicks are instant.

Defines:
  • `infer(images, **kwargs)` → Pi3X result dict
  • `write_colmap(results, image_paths, output_dir, ...)` → COLMAP-format dataset
    (applies metric scaling + confidence filter + subsamples to 2M pts)
  • `save_glb(results, output_dir)` → dense point cloud as .glb
  • `run_full_pipeline(images, output_dir, ...)` → both + gsplat
All params from `example_mm.py` are exposed:
  use_multimodal, use_amp, edge_mask, interval (frame sampling),
  conf_threshold (sigmoid filter), max_points (COLMAP subsample),
  save_glb_export, condition.npz path (multimodal priors).
"""
import sys, os, time, json, pathlib, warnings, shutil
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_DIR = pathlib.Path(os.environ.get('AEI_PI3X_REPO',
                    str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/Pi3X/Pi3'))))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f'  repo        : {REPO_DIR}')
print(f'  exists      : {REPO_DIR.exists()}')
print()

import torch
import numpy as np
print('  Loading pi3 package (one-time cost) ...')
t0 = time.time()
try:
    from pi3.models.pi3x import Pi3X
    print(f'  pi3.models.pi3x.Pi3X ✓')
except Exception as e:
    print(f'  pi3.models.pi3x.Pi3X ✗  {type(e).__name__}: {e}')
    raise
try:
    from pi3.utils.basic import load_images_as_tensor
    print(f'  pi3.utils.basic.load_images_as_tensor ✓')
except Exception as e:
    print(f'  pi3.utils.basic.load_images_as_tensor ✗  {type(e).__name__}: {e}')
    raise
print(f'  total import: {time.time()-t0:.1f}s')
print()

MODEL_REPO_ID = 'yyfz233/Pi3X'   # CC BY-NC-4.0 weights (non-commercial)
MODEL_DIR = drive_root / 'Pi3X-weights'

# ── Lazy model cache ────────────────────────────────────
_MODEL_CACHE = {}
def get_model(force_reload=False, use_multimodal=True):
    """Load Pi3X. use_multimodal=False removes the conditioning branches
    and frees ~2 GB GPU memory — recommended for video (no priors)."""
    key = (use_multimodal,)
    if key in _MODEL_CACHE and not force_reload:
        return _MODEL_CACHE[key]
    print(f'  Loading Pi3X from {MODEL_REPO_ID} (use_multimodal={use_multimodal}) ...')
    t0 = time.time()
    model = Pi3X.from_pretrained(MODEL_REPO_ID, cache_dir=str(drive_root))
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device).eval()
    if not use_multimodal:
        try:
            model.disable_multimodal(free_cuda_cache=True)
            print('  multimodal branches disabled (~2 GB GPU freed)')
        except Exception as e:
            print(f'  WARN: disable_multimodal() failed: {e}')
    _MODEL_CACHE[key] = model
    print(f'  model loaded in {time.time()-t0:.1f}s on {device}')
    return model

# ── Image loading ──────────────────────────────────────
def _collect_images(image_or_dir, max_n=200):
    """Collect a sorted list of image paths from a folder, video, or single image."""
    SUPP = {'.png', '.jpg', '.jpeg'}
    p = pathlib.Path(image_or_dir).resolve()
    if p.is_dir():
        files = sorted(
            x for x in p.rglob('*') if x.suffix.lower() in SUPP
        )
    elif p.is_file() and p.suffix.lower() in SUPP:
        files = [p]
    else:
        raise SystemExit(f'No images found at {p}')
    if not files:
        raise SystemExit(f'No images found at {p}')
    if len(files) > max_n:
        print(f'  WARN: {len(files)} images found; using first {max_n} (T4 VRAM limit)')
        files = files[:max_n]
    return files

def _load_video_frames(video_path, fps=1, max_frames=200):
    """Use ffmpeg to extract frames at the given fps from a video."""
    import subprocess, tempfile
    out_dir = pathlib.Path(tempfile.mkdtemp(prefix='pi3x_video_'))
    out_dir.mkdir(parents=True, exist_ok=True)
    pattern = out_dir / '%05d.png'
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-vf', f'fps={fps},scale=iw:ih',
        '-frames:v', str(max_frames),
        str(pattern),
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'ffmpeg frame extraction failed: {r.stderr[-500:]}')
    files = sorted(out_dir.glob('*.png'))
    print(f'  extracted {len(files)} frames @ {fps} fps from {pathlib.Path(video_path).name}')
    return files

def infer(image_paths, use_multimodal=True, use_amp=True, conf_threshold=0.0,
         minibatch_size=8, edge_mask=True, interval=1, conditions_path=None):
    """Run Pi3X on a list of image paths. Returns the per-view dict with:
      • points       (B, N, H, W, 3)    world-frame (scale-invariant, * metric for true scale)
      • local_points (B, N, H, W, 3)    per-frame local point map
      • rays         (B, N, H, W, 3)    per-pixel ray directions
      • conf         (B, N, H, W, 1)    raw confidence logits (apply sigmoid for [0,1])
      • camera_poses (B, N, 4, 4)       cam2world (OpenCV convention)
      • metric       (B,)               global scale factor (multiply points by this)
    `interval` subsamples every Nth image (upstream default 1 for images, 10 for video).
    `conditions_path` is an optional .npz with pose/intrinsics/depth priors (Pi3X only,
    use_multimodal=True)."""
    model = get_model(use_multimodal=use_multimodal)
    print(f'  loading + preprocessing {len(image_paths)} images ...')
    # load_images_as_tensor handles the upstream preprocessing pipeline.
    imgs = load_images_as_tensor([str(p) for p in image_paths], interval=int(interval)).to('cuda')
    print(f'  images tensor: {tuple(imgs.shape)}  (interval={interval})')
    # Pick AMP dtype (bf16 on Ampere+, fp16 fallback)
    if use_amp and torch.cuda.is_available():
        cap = torch.cuda.get_device_capability()[0]
        amp_dtype = torch.bfloat16 if cap >= 8 else torch.float16
    else:
        amp_dtype = torch.float32
    print(f'  Pi3X forward pass (amp={amp_dtype}) ...')
    t0 = time.time()
    with torch.inference_mode():
        with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
            if conditions_path and use_multimodal and pathlib.Path(conditions_path).exists():
                # Pi3X multimodal: load condition.npz and inject as priors
                try:
                    cond = np.load(conditions_path)
                    conditions = {k: torch.from_numpy(cond[k]).to('cuda').float() for k in cond.files}
                    print(f'  loaded conditions: {list(conditions.keys())}')
                    results = model(imgs[None], conditions=conditions)
                except Exception as e:
                    print(f'  WARN: conditions load failed: {e}; falling back to no priors')
                    results = model(imgs[None])
            else:
                results = model(imgs[None])  # (1, N, 3, H, W) -> dict
    print(f'  inference done in {time.time()-t0:.1f}s')
    # Apply confidence sigmoid for downstream filtering
    if 'conf' in results:
        results['conf_prob'] = torch.sigmoid(results['conf'])
    # Optional edge mask: drop points with normal/depth inconsistency.
    if edge_mask:
        try:
            mask = model.depth_normal_edge(**results)  # (1, N, H, W)
            results['edge_mask'] = mask
            n_kept = mask.float().mean().item() * 100
            print(f'  edge mask: keeping {n_kept:.1f}% of points')
        except Exception as e:
            print(f'  WARN: edge mask failed: {e}')
    # Filter by confidence threshold (applied to per-point sigmoid)
    if conf_threshold > 0 and 'conf_prob' in results:
        conf = results['conf_prob']  # (1, N, H, W, 1)
        n_above = (conf >= conf_threshold).float().mean().item() * 100
        print(f'  conf >= {conf_threshold}: keeping {n_above:.1f}% of points')
        results['conf_mask'] = conf >= conf_threshold
    return results

def write_colmap(results, image_paths, output_dir, max_points=2_000_000):
    """Write Pi3X predictions to COLMAP format.
    COLMAP cameras.txt + images.txt + points3D.ply + a copy of the images.
    Applies Pi3X's `metric` scaling factor for true metric scale, applies the
    confidence mask, and subsamples to `max_points` to keep gsplat happy
    (long videos can produce 10M+ points otherwise)."""
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'images').mkdir(exist_ok=True)
    (output_dir / 'sparse' / '0').mkdir(parents=True, exist_ok=True)
    import shutil
    for i, src in enumerate(image_paths):
        shutil.copy2(str(src), str(output_dir / 'images' / f'{i}.png'))
    # Unpack
    poses = results['camera_poses'][0].cpu().numpy()  # (N, 4, 4) cam2world
    points = results['points'][0].cpu().numpy()        # (N, H, W, 3)
    rays = results['rays'][0].cpu().numpy()            # (N, H, W, 3)
    # Apply Pi3X's global metric scaling factor (per batch)
    if 'metric' in results:
        scale = float(results['metric'][0].cpu())
        points = points * scale
        print(f'  applied metric scale: {scale:.4f}')
    # Apply confidence mask if available
    if 'conf_mask' in results:
        conf_mask = results['conf_mask'][0].cpu().numpy()  # (N, H, W, 1)
        points = points * conf_mask
        n_kept = conf_mask.mean() * 100
        print(f'  conf filter: {n_kept:.1f}% of points kept')
    # Apply edge mask if available
    if 'edge_mask' in results:
        edge_mask = results['edge_mask'][0].cpu().numpy()  # (N, H, W)
        # expand to (N, H, W, 1) for broadcasting
        points = points * edge_mask[..., None]
        print(f'  edge mask: {edge_mask.mean() * 100:.1f}% of points kept')
    n = poses.shape[0]
    H, W = points.shape[1], points.shape[2]
    # Recover intrinsics via the upstream helper
    try:
        from pi3.utils.geometry import recover_intrinsic_from_rays_d
        K = recover_intrinsic_from_rays_d(rays, points / (results['metric'][0].cpu().numpy() if 'metric' in results else 1.0))  # use un-scaled rays
        if K.ndim == 3:
            K = K[0]
    except Exception as e:
        print(f'  WARN: recover_intrinsic_from_rays_d failed: {e}, falling back to identity')
        K = np.array([[max(H, W), 0, W / 2], [0, max(H, W), H / 2], [0, 0, 1]], dtype=np.float32)
    # Cameras.txt (PINHOLE model, 1-indexed)
    cam_lines = []
    for i in range(n):
        fx, fy = float(K[0, 0]), float(K[1, 1])
        cx, cy = float(K[0, 2]), float(K[1, 2])
        cam_lines.append(f'{i+1} PINHOLE {fx:.6f} {fy:.6f} {cx:.6f} {cy:.6f}\n')
    (output_dir / 'sparse' / '0' / 'cameras.txt').write_text(''.join(cam_lines))
    # Images.txt (qw qx qy qz tx ty tz + image_name) — COLMAP expects W2C
    img_lines = []
    from scipy.spatial.transform import Rotation
    for i in range(n):
        c2w = poses[i]
        R_c2w = c2w[:3, :3]
        t_c2w = c2w[:3, 3]
        R_w2c = R_c2w.T
        t_w2c = -R_w2c @ t_c2w
        quat = Rotation.from_matrix(R_w2c).as_quat()  # [x, y, z, w]
        qx, qy, qz, qw = quat
        img_lines.append(
            f'{i+1} {qw:.6f} {qx:.6f} {qy:.6f} {qz:.6f} '
            f'{t_w2c[0]:.6f} {t_w2c[1]:.6f} {t_w2c[2]:.6f} {i+1} {i}.png\n\n'
        )
    (output_dir / 'sparse' / '0' / 'images.txt').write_text(''.join(img_lines))
    # Points3D.ply — flatten all per-frame point clouds + subsample
    flat_pts = points.reshape(-1, 3)
    n_pts = flat_pts.shape[0]
    if n_pts > max_points:
        idx = np.random.choice(n_pts, max_points, replace=False)
        flat_pts = flat_pts[idx]
        print(f'  subsampled: {n_pts:,} -> {max_points:,} points (max_points cap)')
    n_pts = flat_pts.shape[0]
    with open(output_dir / 'sparse' / '0' / 'points3D.ply', 'w') as f:
        f.write('ply\nformat ascii 1.0\n'
                f'element vertex {n_pts}\n'
                'property float x\nproperty float y\nproperty float z\n'
                'end_header\n')
        for p in flat_pts:
            f.write(f'{p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n')
    print(f'  COLMAP dataset written: {n} views, {n_pts:,} points')
    return output_dir

def save_glb(results, image_paths, output_dir, max_pts=200000):
    """Save the dense point cloud as a colorized .glb for direct viewing.
    Mirrors the upstream `example_mm.py --save_path` output style.
    Subsamples to `max_pts` to keep file size reasonable (<50 MB)."""
    try:
        import trimesh
    except ImportError:
        print('  WARN: trimesh not installed, skipping GLB export')
        return None
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    points = results['points'][0].cpu().numpy()  # (N, H, W, 3)
    if 'metric' in results:
        points = points * float(results['metric'][0].cpu())
    flat_pts = points.reshape(-1, 3)
    # Try to use the first image as colors (single-image export)
    if image_paths and pathlib.Path(image_paths[0]).exists():
        try:
            from PIL import Image
            img = Image.open(image_paths[0]).convert('RGB').resize(
                (points.shape[2], points.shape[1]))
            cols = np.array(img).reshape(-1, 3)
        except Exception:
            cols = np.ones_like(flat_pts) * 128
    else:
        cols = np.ones_like(flat_pts) * 128
    if len(flat_pts) > max_pts:
        idx = np.random.choice(len(flat_pts), max_pts, replace=False)
        flat_pts = flat_pts[idx]
        cols = cols[idx]
    pc = trimesh.PointCloud(vertices=flat_pts, colors=cols.astype(np.uint8))
    glb_path = output_dir / 'reconstruction.glb'
    pc.export(str(glb_path))
    print(f'  GLB exported: {glb_path}  ({glb_path.stat().st_size//(1024*1024)} MB, {len(flat_pts):,} pts)')
    return glb_path

def train_gsplat(colmap_dir, output_dir, iterations=7000):
    """Train a 3DGS scene from a COLMAP dataset using the gsplat simple_trainer."""
    from gsplat.examples.simple_trainer import main as gsplat_main
    from argparse import Namespace
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'  gsplat training ({iterations} iters) ...')
    t0 = time.time()
    args = Namespace(
        data_dir=str(colmap_dir),
        result_dir=str(output_dir),
        data_factor=1,
        init_scale=0.1,
        steps=int(iterations),
        means_l2norm_reg=0.0,
        sh_degree=3,
        sh_degree_interval=1000,
        packed=False,
        antialiased=False,
        eval_steps=[-1],
        save_steps=[int(iterations)],
        disable_viewer=True,
        max_n_render_parallel=8,
    )
    try:
        gsplat_main(args)
    except SystemExit:
        pass
    ply_candidates = sorted(output_dir.rglob('*.ply'))
    ply = max(ply_candidates, key=lambda p: p.stat().st_mtime) if ply_candidates else None
    if ply is None:
        raise FileNotFoundError(f'gsplat did not produce a .ply in {output_dir}')
    print(f'  gsplat training done in {time.time()-t0:.0f}s')
    print(f'  output .ply: {ply}  ({ply.stat().st_size//(1024*1024)} MB)')
    return ply

def _mirror_to_drive(paths, output_root, drive_subdir='Pi3X'):
    if not paths:
        return
    drive_base = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out') / drive_subdir
    try:
        drive_base.parent.mkdir(parents=True, exist_ok=True)
        for src in paths:
            try:
                _src = pathlib.Path(src) if not isinstance(src, pathlib.Path) else src
                dst = drive_base / _src.relative_to(pathlib.Path(output_root))
                dst.parent.mkdir(parents=True, exist_ok=True)
                if _src.resolve() == dst.resolve():
                    continue
                if dst.exists() and dst.stat().st_size == _src.stat().st_size:
                    continue
                shutil.copy2(str(_src), str(dst))
            except Exception as e:
                print(f'  WARN: drive mirror of {_src.name} failed: {e}')
        print(f'  drive mirror: {drive_base}')
    except Exception as e:
        print(f'  drive mirror skipped: {e}')

def run_full_pipeline(image_paths, output_dir, train_3dgs=True,
                     gsplat_iterations=7000, use_multimodal=True,
                     use_amp=True, conf_threshold=0.0,
                     minibatch_size=8, edge_mask=True, interval=1,
                     conditions_path=None, max_points=2_000_000,
                     save_glb_export=True, do_drive_save=True):
    """Full pipeline: Pi3X → COLMAP → gsplat → .ply (+ optional .glb)
    Returns (colmap_dir, ply_path, glb_path, n_views)"""
    image_paths = [pathlib.Path(p) for p in image_paths]
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    results = infer(image_paths, use_multimodal=use_multimodal,
                   use_amp=use_amp, conf_threshold=conf_threshold,
                   minibatch_size=minibatch_size, edge_mask=edge_mask,
                   interval=interval, conditions_path=conditions_path)
    n = results['camera_poses'].shape[1]
    colmap_dir = write_colmap(results, image_paths, output_dir / 'colmap',
                              max_points=max_points)
    glb_path = None
    if save_glb_export:
        glb_path = save_glb(results, image_paths, output_dir)
    ply = None
    if train_3dgs:
        ply = train_gsplat(colmap_dir, output_dir / 'gsplat',
                          iterations=gsplat_iterations)
    if do_drive_save and ply is not None:
        paths_to_save = [ply]
        if glb_path is not None:
            paths_to_save.append(glb_path)
        _mirror_to_drive(paths_to_save, str(output_dir / 'gsplat'))
    return colmap_dir, ply, glb_path, n

print('  pipeline ready: infer, write_colmap, train_gsplat, run_full_pipeline')


In [ ]:
#@title STEP 3 — Core helpers (single scene, batch, video frames)
"""
Re-exports the pipeline from STEP 2 and adds:
  • `run_single_scene(image_or_video_or_dir, output_dir, ...)` — convenience
  • `run_batch(input_dir, output_dir, ...)` — process multiple subdirs
  • `extract_video_frames(video_path, output_dir, fps)` — already imported
"""
import subprocess, json

def run_single_scene(image_or_video, output_dir, train_3dgs=True,
                     gsplat_iterations=7000, video_fps=1, max_frames=200,
                     use_multimodal=True, use_amp=True, conf_threshold=0.0,
                     minibatch_size=8, edge_mask=True, interval=1,
                     conditions_path=None, max_points=2_000_000,
                     save_glb_export=True, do_drive_save=True):
    """Run the full Pi3X + gsplat pipeline on one image folder OR one video.
    Auto-detects which by file extension."""
    src = pathlib.Path(image_or_video).resolve()
    out = pathlib.Path(output_dir).resolve()
    out.mkdir(parents=True, exist_ok=True)
    if src.suffix.lower() in {'.mp4', '.mov', '.webm', '.avi', '.mkv', '.m4v'}:
        frames_dir = out / 'frames'
        frames_dir.mkdir(parents=True, exist_ok=True)
        if not any(frames_dir.glob('*.png')):
            image_paths = _load_video_frames(src, fps=video_fps, max_frames=max_frames)
        else:
            image_paths = sorted(frames_dir.glob('*.png'))
    else:
        if src.is_dir():
            image_paths = _collect_images(src)
        elif src.is_file():
            image_paths = [src]
        else:
            raise SystemExit(f'Input not found: {src}')
    colmap_dir, ply, glb_path, n = run_full_pipeline(
        image_paths, out,
        train_3dgs=train_3dgs,
        gsplat_iterations=gsplat_iterations,
        use_multimodal=use_multimodal,
        use_amp=use_amp,
        conf_threshold=conf_threshold,
        minibatch_size=minibatch_size,
        edge_mask=edge_mask,
        interval=interval,
        conditions_path=conditions_path,
        max_points=max_points,
        save_glb_export=save_glb_export,
        do_drive_save=do_drive_save,
    )
    return ply

def run_batch(input_dir, output_dir, train_3dgs=True, gsplat_iterations=7000,
              video_fps=1, max_frames=200, use_multimodal=True,
              use_amp=True, conf_threshold=0.0, minibatch_size=8,
              edge_mask=True, interval=1, conditions_path=None,
              max_points=2_000_000, save_glb_export=True,
              do_drive_save=True):
    """Process every video or subdirectory of `input_dir` as a separate scene."""
    in_dir = pathlib.Path(input_dir).resolve()
    if not in_dir.exists():
        raise SystemExit(f'Input not found: {in_dir}')
    out_dir = pathlib.Path(output_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    SUPP = {'.mp4', '.mov', '.webm', '.avi', '.mkv', '.m4v'}
    videos = sorted(p for p in in_dir.glob('*') if p.suffix.lower() in SUPP)
    subdirs = sorted(p for p in in_dir.iterdir() if p.is_dir() and any(p.iterdir()))
    scenes = [(p, p.stem) for p in videos] + [(p, p.name) for p in subdirs]
    if not scenes:
        raise SystemExit(f'No videos or subdirs found in {in_dir}')
    n_ok = 0
    for src, slug in scenes:
        scene_out = out_dir / slug
        scene_out.mkdir(parents=True, exist_ok=True)
        existing_plys = list(scene_out.rglob('gsplat/*.ply'))
        if existing_plys and existing_plys[0].stat().st_size > 1024:
            print(f'  skip: {slug}  (already done)')
            continue
        try:
            run_single_scene(src, scene_out,
                             train_3dgs=train_3dgs,
                             gsplat_iterations=gsplat_iterations,
                             video_fps=video_fps, max_frames=max_frames,
                             use_multimodal=use_multimodal,
                             use_amp=use_amp,
                             conf_threshold=conf_threshold,
                             minibatch_size=minibatch_size,
                             edge_mask=edge_mask,
                             interval=interval,
                             conditions_path=conditions_path,
                             max_points=max_points,
                             save_glb_export=save_glb_export,
                             do_drive_save=False)
            n_ok += 1
        except Exception as e:
            print(f'  ERROR on {slug}: {type(e).__name__}: {e}')
    if do_drive_save and n_ok > 0:
        all_plys = sorted(out_dir.rglob('gsplat/*.ply'))
        all_glbs = sorted(out_dir.rglob('reconstruction.glb'))
        paths = all_plys + all_glbs
        if paths:
            _mirror_to_drive(paths, str(out_dir))
    print(f'\n  done: {n_ok} new scene(s) processed')
    return n_ok

print('  scene helpers ready: run_single_scene, run_batch')


In [ ]:
#@title STEP 4 — Launch Gradio UI
"""
Interactive Gradio UI:
  - Multi-file upload (any 1+ images) OR video upload
  - All Pi3X params exposed (use_multimodal, use_amp, edge_mask, interval,
    conf_threshold, conditions_path, max_points, save_glb_export)
  - Auto-runs gsplat after Pi3X for 3DGS output
  - Run button -> .ply + optional .glb outputs
  - Drive mirror toggle
  - BSD-3 + CC BY-NC-4.0 license notice
"""
import os, sys, time, json, shutil, pathlib, traceback, tempfile
import gradio as gr

gr.TEMP_DIR = '/tmp/gradio_pi3x'
os.makedirs(gr.TEMP_DIR, exist_ok=True)

def _do_run(files, video_file, gsplat_iterations, use_multimodal,
            use_amp, edge_mask, conf_threshold, interval,
            conditions_path, max_points, save_glb_export,
            video_fps, max_frames, do_drive_save,
            progress=gr.Progress()):
    try:
        tmp = pathlib.Path(tempfile.mkdtemp(prefix='pi3x_gradio_'))
        if video_file is not None:
            src = pathlib.Path(video_file if isinstance(video_file, str) else video_file.name)
            progress(0.1, 'Extracting video frames ...')
            image_paths = _load_video_frames(src, fps=int(video_fps), max_frames=int(max_frames))
        elif files:
            saved = []
            for f in files:
                src = pathlib.Path(f.name if hasattr(f, 'name') else f)
                dst = tmp / src.name
                shutil.copy2(str(src), str(dst))
                saved.append(str(dst))
            image_paths = _collect_images(tmp, max_n=int(max_frames))
        else:
            return 'Please upload a video or images.', None, None, None
        n_imgs = len(image_paths)
        progress(0.25, f'Running Pi3X on {n_imgs} frames (permutation-equivariant) ...')
        colmap_dir, ply, glb_path, n_views = run_full_pipeline(
            image_paths, tmp,
            train_3dgs=True,
            gsplat_iterations=int(gsplat_iterations),
            use_multimodal=use_multimodal,
            use_amp=use_amp,
            edge_mask=edge_mask,
            conf_threshold=float(conf_threshold),
            interval=int(interval),
            conditions_path=(conditions_path.strip() or None),
            max_points=int(max_points),
            save_glb_export=save_glb_export,
            do_drive_save=do_drive_save,
        )
        if ply is None:
            return 'Pipeline failed - see cell output for details.', None, None, None
        progress(1.0, 'Done')
        status = (f'Generated 3DGS scene from {n_views} frames -> '
                  f'{ply.name}  ({ply.stat().st_size//(1024*1024)} MB)'
                  + (f'  +  reconstruction.glb' if glb_path else ''))
        return status, str(ply), _splat_viewer_html(str(ply), str(glb_path) if glb_path else None), (str(glb_path) if glb_path else None)
    except Exception as e:
        traceback.print_exc()
        return f'Error: {type(e).__name__}: {e}', None, None, None

def _splat_viewer_html(ply_path, glb_path=None):
    if not ply_path or not pathlib.Path(ply_path).exists():
        return '<p style="color:#888">No scene to preview</p>'
    p = pathlib.Path(ply_path)
    glb_html = ''
    if glb_path and pathlib.Path(glb_path).exists():
        gp = pathlib.Path(glb_path)
        glb_html = (f'<br><br>GLB: {gp.name}  ({gp.stat().st_size//(1024*1024)} MB) - '
                    f'<a href="https://playcanvas.com/viewer" target="_blank" '
                    f'style="color:#4af">playcanvas.com/viewer</a>')
    return (f'<div style="width:100%; height:240px; background:#1a1a1a; display:flex; '
            f'align-items:center; justify-content:center; color:#aaa; font-family:monospace; '
            f'padding:20px; text-align:center; border-radius:8px">'
            f'3DGS scene ready: {p.name}  ({p.stat().st_size//(1024*1024)} MB)'
            f'<br><small>Open the .ply in: '
            f'<a href="https://supersplat.dev" target="_blank" style="color:#4af">supersplat.dev</a> - '
            f'<a href="https://playcanvas.com/viewer" target="_blank" style="color:#4af">playcanvas.com/viewer</a> - '
            f'<a href="https://gsplat.tech" target="_blank" style="color:#4af">gsplat.tech</a>'
            f'{glb_html}'
            f'</small></div>')

with gr.Blocks(title='Pi3X - Video-Native 3DGS (BSD-3 + CC BY-NC-4.0)') as demo:
    gr.Markdown(value=
        '## [Pi3 / Pi3X](https://yyfz.github.io/pi3/) - Video-Native 3DGS from any order\n'
        'Shanghai AI Lab + ZJU permutation-equivariant 3D reconstruction (ICLR 2026).\n'
        'No reference view, no drift on long videos, frame order is irrelevant.\n'
        '* **Paper:** [arXiv 2507.13347](https://arxiv.org/abs/2507.13347) (ICLR 2026) | '
          '**Code:** [yyfz/Pi3](https://github.com/yyfz/Pi3) (BSD-3) | '
          '**Weights:** [yyfz233/Pi3X](https://huggingface.co/yyfz233/Pi3X) (CC BY-NC-4.0, non-commercial)'
    )
    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(
                label='Images  ( 1+ .png / .jpg, any order )',
                file_count='multiple',
                file_types=['.png', '.jpg', '.jpeg'],
            )
            video_file = gr.Video(
                label='...or upload a video  ( .mp4 / .mov / .webm )',
                sources=['upload'],
            )
            with gr.Accordion('Generation Settings', open=False):
                gsplat_iterations = gr.Slider(
                    1000, 30000, value=7000, step=1000,
                    label='gsplat iterations (3DGS training)',
                    info='Number of gsplat training iterations. 7000 is a good balance.'
                )
                use_multimodal = gr.Checkbox(
                    False,
                    label='use_multimodal (conditioning on pose/intrinsics/depth)',
                    info='Enable conditioning branches. Off = video-native no-prior mode (frees ~2 GB GPU). Recommended for unordered video frames.'
                )
                conditions_path = gr.Textbox(
                    value='',
                    label='conditions.npz path (optional, Pi3X multimodal only)',
                    info='Path to a .npz with pose/intrinsics/depth priors (see upstream example_mm.py). Only used when use_multimodal=True. Leave empty to use image-only.'
                )
                use_amp = gr.Checkbox(
                    True,
                    label='use_amp (bf16/fp16 mixed precision)',
                    info='Use automatic mixed precision. bf16 on Ampere+, fp16 fallback. ~2x speedup.'
                )
                edge_mask = gr.Checkbox(
                    True,
                    label='edge_mask (depth-normal edge filter)',
                    info='Mask out edge artifacts using normal/depth consistency. Improves 3DGS quality.'
                )
                conf_threshold = gr.Slider(
                    0.0, 0.95, value=0.0, step=0.05,
                    label='conf_threshold (sigmoid filter)',
                    info='Drop points with sigmoid(conf) below this. 0 = no filter, 0.5 = drop <50% confidence, 0.9 = keep only top 10%.'
                )
                interval = gr.Slider(
                    1, 60, value=1, step=1,
                    label='interval (frame sampling)',
                    info='Subsample every Nth image before inference. 1 = use all. 10 = upstream default for video (use for long clips).'
                )
                max_points = gr.Slider(
                    100_000, 5_000_000, value=2_000_000, step=100_000,
                    label='max_points (COLMAP point cloud cap)',
                    info='Max points in points3D.ply. Long videos can produce 10M+ points; gsplat prefers <=2M. Lower = faster training, less detail.'
                )
                save_glb_export = gr.Checkbox(
                    True,
                    label='Also save .glb (dense point cloud)',
                    info='Export a colorized .glb of the dense point cloud. ~50 MB. Open in playcanvas.com/viewer or any glTF viewer.'
                )
                video_fps = gr.Slider(
                    0.5, 5, value=1, step=0.5,
                    label='Video frame rate (extracted)',
                    info='Frames per second to extract from the video. 1 fps is a good default for long videos.'
                )
                max_frames = gr.Slider(
                    4, 200, value=64, step=4,
                    label='Max frames',
                    info='Maximum number of frames to process. Lower = less VRAM. Pi3X is permutation-equivariant so all frames are weighted equally.'
                )
                do_drive_save = gr.Checkbox(
                    True,
                    label='Mirror .ply to Google Drive',
                    info='Copy outputs to /content/drive/MyDrive/AEI_3D_Out/Pi3X/'
                )
            run_btn = gr.Button('Run', variant='primary')
        with gr.Column(scale=1):
            log = gr.Textbox(label='Status', lines=2, interactive=False)
            output = gr.File(label='Generated 3DGS .ply', interactive=False)
            glb_output = gr.File(label='Dense point cloud .glb (optional)', interactive=False)
            viewer = gr.HTML(label='3DGS Preview')
            gr.Markdown(
                '**Next steps for the .ply**\n'
                '* Open in [supersplat.dev](https://supersplat.dev) for cleanup + editing\n'
                '* Open the .glb in [playcanvas.com/viewer](https://playcanvas.com/viewer) for a quick 3D preview\n'
                '* Pipe into **SplatTransform_Colab** STEP 3 to compress to game-ready SOG/SPLAT/SPZ/GLB\n'
                '* Or load directly in Three.js / WebGPU'
            )
    run_btn.click(
        _do_run,
        inputs=[files, video_file, gsplat_iterations, use_multimodal,
                use_amp, edge_mask, conf_threshold, interval,
                conditions_path, max_points, save_glb_export,
                video_fps, max_frames, do_drive_save],
        outputs=[log, output, viewer, glb_output],
    )
    def _welcome():
        return ('Ready. Upload 1+ images or a short video. Frame order is '
                'irrelevant - Pi3X is permutation-equivariant.')
    demo.load(_welcome, outputs=[log])

from IPython.display import clear_output
clear_output()
demo.queue(default_concurrency_limit=2).launch(
    share=False, prevent_thread_lock=True, inline=True,
    show_error=True, height=1100,
)
print('\n  Gradio UI running above (cell stays alive - do not re-run)')

In [ ]:
#@title STEP 5 — Keep alive + session summary
"""
Keeps the runtime alive for 12 h. Prints a session summary.
"""
import datetime, time, json, sys, pathlib, os, shutil
summary = {
    'timestamp'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'python'      : sys.version.split()[0],
    'torch'       : None, 'cuda': None, 'gpus': [],
    'repo'        : str(REPO_DIR),
    'ffmpeg'      : shutil.which('ffmpeg') is not None,
    'model_cached': list(_MODEL_CACHE.keys()),
    'drive_cache' : str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/Pi3X')),
}
try:
    import torch
    summary['torch'] = torch.__version__
    summary['cuda']  = torch.version.cuda
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            summary['gpus'].append(f'{p.name}  {p.total_memory // (1024**3)} GB')
except Exception as e:
    summary['torch_error'] = str(e)
print(json.dumps(summary, indent=2))
print()
print('  • STEP 4 is the Gradio UI (interactive)')
print('  • STEP 6 is the Colab single-scene picker')
print('  • STEP 7 is the Colab batch (multiple scenes)')
print('  • Outputs land in OUTPUT_DIR and (if mirrored) in /content/drive/MyDrive/AEI_3D_Out/Pi3X/')
print()
print('  License: BSD-3 (code) + CC BY-NC-4.0 (weights) — non-commercial use only.')
print()
print('  Cell will keep the runtime alive for 12 h unless you disconnect.')
print('  Press the stop button in the toolbar to release the GPU.')

_start = time.time()
while time.time() - _start < 43200:
    time.sleep(300)
    print(f'  [{int(time.time()-_start)//60} min] still running — close tab to stop')


In [ ]:
#@title STEP 6 — Quick test (Colab single-scene picker)
"""
Quick-test one video OR one image folder. Use this for verification
or for scripting (no Gradio UI). For interactive UI, use STEP 4.
"""
import time, pathlib

INPUT_PATH       = ''  #@param {type:'string', placeholder: '/content/my_video.mp4 or /content/my_photos/'}  # info='Either a video file or a folder of images.'
OUTPUT_DIR       = '/content/Pi3X_Out'  #@param {type:'string'}  # info='Where the COLMAP dataset + .ply + .glb will be written.'

GSPLAT_ITERATIONS = 7000  #@param {type:'slider', min:1000, max:30000, step:1000}  # info='Number of gsplat training iterations. 7000 is a good balance.'
USE_MULTIMODAL   = False  #@param {type:'boolean'}  # info='Enable conditioning on pose/intrinsics/depth. Off = video-native no-prior mode (frees ~2 GB GPU).'
CONDITIONS_PATH  = ''  #@param {type:'string', placeholder: '/content/my_scene/condition.npz'}  # info='Optional .npz with pose/intrinsics/depth priors. Only used when USE_MULTIMODAL=True.'
USE_AMP          = True  #@param {type:'boolean'}  # info='bf16/fp16 mixed precision. ~2x speedup on Ampere+ GPUs.'
EDGE_MASK        = True  #@param {type:'boolean'}  # info='Mask out edge artifacts using normal/depth consistency.'
CONF_THRESHOLD   = 0.0  #@param {type:'slider', min:0, max:0.95, step:0.05}  # info='Drop points with sigmoid(conf) below this. 0 = no filter.'
INTERVAL         = 1  #@param {type:'slider', min:1, max:60, step:1}  # info='Subsample every Nth image. 1 = use all. 10 = upstream video default for long clips.'
MAX_POINTS       = 2_000_000  #@param {type:'slider', min:100000, max:5000000, step:100000}  # info='Max points in points3D.ply. Long videos can produce 10M+.'
SAVE_GLB         = True  #@param {type:'boolean'}  # info='Also export a colorized .glb of the dense point cloud.'
VIDEO_FPS        = 1  #@param {type:'slider', min:0.5, max:5, step:0.5}  # info='Frames per second to extract from the video.'
MAX_FRAMES       = 64  #@param {type:'slider', min:4, max:200, step:4}  # info='Maximum number of frames to process.'
DO_DRIVE_SAVE    = True  #@param {type:'boolean'}  # info='Copy outputs to /content/drive/MyDrive/AEI_3D_Out/Pi3X/.'

if not INPUT_PATH.strip():
    raise SystemExit('Set INPUT_PATH to a video or a folder of images.')
src = pathlib.Path(INPUT_PATH).resolve()
if not src.exists():
    raise SystemExit(f'Input not found: {src}')
out = pathlib.Path(OUTPUT_DIR).resolve()
out.mkdir(parents=True, exist_ok=True)
print(f'  input    : {src}')
print(f'  output   : {out}')
print(f'  iters    : {GSPLAT_ITERATIONS}')
print()
t0 = time.time()
colmap_dir, ply, glb_path, n = run_full_pipeline(
    [src], out,
    train_3dgs=True,
    gsplat_iterations=GSPLAT_ITERATIONS,
    use_multimodal=USE_MULTIMODAL,
    conditions_path=(CONDITIONS_PATH.strip() or None),
    use_amp=USE_AMP,
    edge_mask=EDGE_MASK,
    conf_threshold=CONF_THRESHOLD,
    interval=INTERVAL,
    max_points=MAX_POINTS,
    save_glb_export=SAVE_GLB,
    video_fps=VIDEO_FPS, max_frames=MAX_FRAMES,
    do_drive_save=DO_DRIVE_SAVE,
)
elapsed = time.time() - t0
print(f'\n  done in {elapsed:.0f}s ({n} views)')
print(f'  output: {ply}  ({ply.stat().st_size//(1024*1024)} MB)')
if glb_path:
    print(f'  glb   : {glb_path}  ({glb_path.stat().st_size//(1024*1024)} MB)')

In [ ]:
#@title STEP 7 — Batch process multiple scenes
"""
Process every video or subdirectory in `INPUT_DIR` as a separate scene.
Each output scene has its own .ply + .glb. Already-done scenes are skipped.
"""
import time, pathlib

INPUT_DIR        = ''  #@param {type:'string', placeholder: '/content/my_scenes/'}  # info='Folder containing videos or scene subdirs.'
OUTPUT_DIR       = '/content/Pi3X_Out'  #@param {type:'string'}  # info='Where each scene's .ply + .glb will be written.'

GSPLAT_ITERATIONS = 7000  #@param {type:'slider', min:1000, max:30000, step:1000}  # info='gsplat training iterations per scene.'
USE_MULTIMODAL   = False  #@param {type:'boolean'}  # info='Enable conditioning on pose/intrinsics/depth. Off = video-native no-prior mode.'
CONDITIONS_PATH  = ''  #@param {type:'string'}  # info='Optional .npz with pose/intrinsics/depth priors per scene.'
USE_AMP          = True  #@param {type:'boolean'}  # info='bf16/fp16 mixed precision.'
EDGE_MASK        = True  #@param {type:'boolean'}  # info='Mask out edge artifacts using normal/depth consistency.'
CONF_THRESHOLD   = 0.0  #@param {type:'slider', min:0, max:0.95, step:0.05}  # info='Drop points with sigmoid(conf) below this.'
INTERVAL         = 1  #@param {type:'slider', min:1, max:60, step:1}  # info='Subsample every Nth image. 1 = use all.'
MAX_POINTS       = 2_000_000  #@param {type:'slider', min:100000, max:5000000, step:100000}  # info='Max points in points3D.ply per scene.'
SAVE_GLB         = True  #@param {type:'boolean'}  # info='Also export a colorized .glb per scene.'
VIDEO_FPS        = 1  #@param {type:'slider', min:0.5, max:5, step:0.5}  # info='Frames per second to extract from videos.'
MAX_FRAMES       = 64  #@param {type:'slider', min:4, max:200, step:4}  # info='Maximum number of frames per scene.'
DO_DRIVE_SAVE    = True  #@param {type:'boolean'}  # info='Copy all scene .ply + .glb files to /content/drive/MyDrive/AEI_3D_Out/Pi3X/.'

if not INPUT_DIR.strip():
    raise SystemExit('Set INPUT_DIR to a folder of videos or subdirectories.')
in_dir = pathlib.Path(INPUT_DIR).resolve()
if not in_dir.exists():
    raise SystemExit(f'Input not found: {in_dir}')
out_dir = pathlib.Path(OUTPUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
print(f'  input    : {in_dir}')
print(f'  output   : {out_dir}')
print()
t0 = time.time()
n = run_batch(
    in_dir, out_dir,
    gsplat_iterations=GSPLAT_ITERATIONS,
    use_multimodal=USE_MULTIMODAL,
    conditions_path=(CONDITIONS_PATH.strip() or None),
    use_amp=USE_AMP,
    edge_mask=EDGE_MASK,
    conf_threshold=CONF_THRESHOLD,
    interval=INTERVAL,
    max_points=MAX_POINTS,
    save_glb_export=SAVE_GLB,
    video_fps=VIDEO_FPS, max_frames=MAX_FRAMES,
    do_drive_save=DO_DRIVE_SAVE,
)
elapsed = time.time() - t0
print(f'\n  batch done: {n} scene(s) in {elapsed:.0f}s')